# CryptoForecast — End-to-End Demo
This notebook demonstrates the full workflow:
1. Ingest data from CoinGecko
2. Train a Prophet forecast model
3. Generate forecasts for every historical + future time point
4. Visualise results interactively with Plotly

In [ ]:
%pip install -e .. -q

In [ ]:
from cryptoforecast.utils.logger import setup_logger
setup_logger()

from cryptoforecast.ingestion.fetcher import CryptoFetcher
from cryptoforecast.storage.database import CryptoDatabase
from cryptoforecast.modeling.trainer import ForecastTrainer
from cryptoforecast.forecasting.predictor import ForecastPredictor
from cryptoforecast.utils.visualizer import ForecastVisualizer

## Step 1 — Ingest Data

In [ ]:
COINS = ["bitcoin", "ethereum", "solana"]

fetcher = CryptoFetcher()
db = CryptoDatabase()  # writes to data/crypto.duckdb

data = fetcher.fetch_all(coin_ids=COINS, days=365)
db.upsert_market_prices(data['market_charts'])
db.upsert_ohlcv(data['ohlcv'])
db.upsert_market_snapshot(data['snapshot'])

db.summary()

## Step 2 — Train Forecast Models

In [ ]:
trainer = ForecastTrainer(db=db, model_name='prophet')
models = trainer.train_all(coin_ids=COINS)

# Print metrics
for coin, m in models.items():
    print(f"{coin}: test_MAE={m.metrics['test_mae']:.2f}, test_MAPE={m.metrics['test_mape']:.2%}")

## Step 3 — Generate Forecasts

In [ ]:
predictor = ForecastPredictor(db=db, model_name='prophet', horizon=30)
forecasts = predictor.forecast_all(coin_ids=COINS)

# Preview Bitcoin forecast
btc_fc = forecasts['bitcoin']
btc_fc[btc_fc['is_future']].head(10)

## Step 4 — Visualise

In [ ]:
viz = ForecastVisualizer()

# Single-coin forecast chart
fig = viz.plot_forecast(btc_fc, coin_id='bitcoin')
fig.show()

In [ ]:
# Multi-coin comparison (normalised to 100)
fig2 = viz.plot_multi_coin(forecasts, normalise=True)
fig2.show()

## Pipeline Shortcut — one function call

In [ ]:
from cryptoforecast.pipeline import run_pipeline

results = run_pipeline(
    coins=["bitcoin"],
    days=180,
    model_name="ensemble",
    horizon=30,
)
results['bitcoin'].tail()